In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [14]:
def get_clean_report_text(s):
    full_text = s.get_text(separator="\n")
    
    start_markers = [
        "HISTORY OF PRESENT ILLNESS",
        "CHIEF COMPLAINT",
        "REASON FOR VISIT",
        "SUBJECTIVE",
        "EMERGENCY DEPARTMENT",
        "PRESENTING COMPLAINT",
    ]
    end_markers = [
        "About This Sample",
        "Go Back to Emergency Room",
        "Related Samples",
        "Keywords:",
        "Educational Disclaimer",
    ]
    
    start_idx = None
    for marker in start_markers:
        idx = full_text.find(marker)
        if idx != -1:
            if start_idx is None or idx < start_idx:
                start_idx = idx

    end_idx = None
    for marker in end_markers:
        idx = full_text.find(marker)
        if idx != -1:
            if end_idx is None or idx < end_idx:
                end_idx = idx

    if start_idx is None:
        return None

    if end_idx is None or end_idx < start_idx:
        return full_text[start_idx:].strip()

    return full_text[start_idx:end_idx].strip()

In [15]:
# ── Option 2: scrape ER reports directly (more efficient) ────────────────────


def get_er_samples():
    i=0
    records = []
    url = "https://www.mtsamples.com/site/pages/browse.asp?type=93-Emergency+Room+Reports"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    links = soup.find_all("a", href=True)
    sample_links = [
        l["href"] for l in links
        if "sample.asp" in l["href"] and "Emergency" in l["href"]
    ]
    print(f"Found {len(sample_links)} ER sample links")

    for link in sample_links:
        r = requests.get(link)
        s = BeautifulSoup(r.text, "html.parser")
        try:
            # Extract clean title from the Sample parameter in the URL
            sample_param = link.split("Sample=")[-1]
            title = sample_param.split("-", 1)[-1].replace("+", " ").replace("%2F", "/")

            text = get_clean_report_text(s)
            records.append({"title": title, "text": text})
            print(f"OK: {title}")
            i+=1
            print(i)
        except Exception as e:
            print(f"Skipping {link}: {e}")
        time.sleep(0.5)

    return pd.DataFrame(records)

In [ ]:
# ── 2. Load and explore ───────────────────────────────────────────────────────
df_er = get_er_samples()
print(f"\nTotal ER reports collected: {len(df_er)}")

Found 225 ER sample links
OK: Abdominal Pain - Consult
1
OK: Abrasions & Lacerations - ER Visit
2
OK: Accidental Celesta Ingestion - ER Visit
3
OK: Acute Inferior Myocardial Infarction
4
OK: Agitation - ER Visit
5
OK: Air Under Diaphragm - Consult
6
OK: Airway Compromise & Foreign Body - ER Visit
7
OK: Altered Mental Status - ER Visit
8
OK: Angina - Consult
9
OK: Ankle pain
10
OK: Ant Bait Exposure - ER Visit
11
OK: Asthma in a 5-year-old
12
OK: Blood In Toilet
13
OK: Blood in Urine - ER Visit
14
OK: Blood per Rectum
15
OK: Bronchiolitis - 2-month-old
16
OK: Chest Tube Insertion in ER
17
OK: Closed Head Injury
18
OK: Colostomy Failure
19
OK: Consult - ICU Management
20
OK: Consult/ER Report - OB/GYN
21
OK: Cut on Foot - ER Visit
22
OK: CVA Consult - ER Visit
23
OK: Dental Pain
24
OK: Dental Pain - Emergency Visit
25
OK: Difficulty Breathing - ER Visit 
26
OK: Ecstasy Ingestion - ER Visit
27
OK: ER Report - Chest Pain
28
OK: ER Report - Chest Pain & Fever
29
OK: ER Report - COPD
30
OK: 

### Save data in a folder to explore

In [7]:
import os
import json
import re

# ── 1. Save as CSV ────────────────────────────────────────────────────────────
output_dir = "mtsamples_er"
os.makedirs(output_dir, exist_ok=True)

# Save the full dataframe as CSV
df_er.to_csv(os.path.join(output_dir, "er_reports.csv"), index=False)
print(f"Saved {len(df_er)} reports to {output_dir}/er_reports.csv")

# ── 2. Optionally save each report as its own .txt file ──────────────────────
txt_dir = os.path.join(output_dir, "texts")
os.makedirs(txt_dir, exist_ok=True)

for _, row in df_er.iterrows():
    # Clean the title: strip whitespace/newlines, then sanitize for filename
    clean_title = " ".join(row['title'].split())  # collapses all whitespace and newlines
    clean_title = re.sub(r'Sample Name:\s*', '', clean_title)  # remove "Sample Name:" prefix
    clean_title = re.sub(r'Medical Specialty:.*', '', clean_title).strip()  # remove everything from "Medical Specialty:" onward
    clean_title = re.sub(r'[^\w\s-]', '', clean_title)  # remove special chars like &
    filename = clean_title.replace(" ", "_") + ".txt"
    
    filepath = os.path.join(txt_dir, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(row['text'] or "")

print(f"Saved individual .txt files to {txt_dir}/")

# Verify the filenames look clean
import os
for f in sorted(os.listdir(txt_dir))[:5]:
    print(f)





Saved 225 reports to mtsamples_er/er_reports.csv
Saved individual .txt files to mtsamples_er/texts/
.ipynb_checkpoints
Abdominal_Pain_-_Consult.txt
Abdominal_Pain_2D_Consult.txt
Abrasions_26_Lacerations_2D_ER_Visit.txt
Abrasions__Lacerations_-_ER_Visit.txt


In [8]:
# ── 3. Explore ────────────────────────────────────────────────────────────────
# Basic stats
print(f"\nTotal reports:   {len(df_er)}")
print(f"Missing texts:   {df_er['text'].isna().sum()}")
print(f"Avg text length: {df_er['text'].str.len().mean():.0f} chars")

# Look at a random example
sample = df_er.sample(1).iloc[0]
print(f"\n── Random example ──────────────────────────────")
print(f"Title: {sample['title']}")
print(f"Text:\n{sample['text'][:1000]}...")


Total reports:   225
Missing texts:   36
Avg text length: 0 chars

── Random example ──────────────────────────────
Title: Motor Vehicle Accident
Text:
...


In [11]:
    
# Test on one single page
test_url = "https://www.mtsamples.com/site/pages/sample.asp?Type=93-Emergency+Room+Reports&Sample=2294-Abrasions+%26+Lacerations+-+ER+Visit"
r = requests.get(test_url)
s = BeautifulSoup(r.text, "html.parser")

text = get_clean_report_text(s)
print(f"Extracted {len(text) if text else 0} chars")
print(text[:3000] if text else "EMPTY")

  start_idx: 11779, end_idx: 10821
Extracted 5768 chars
HISTORY OF PRESENT ILLNESS: 
 This is a 12-year-old male, who was admitted to the Emergency Department, who fell off his bicycle, not wearing a helmet, a few hours ago.  There was loss of consciousness.  The patient complains of neck pain.
CHRONIC/INACTIVE CONDITIONS:
  None.
PERSONAL/FAMILY/SOCIAL HISTORY/ILLNESSES:
  None.
PREVIOUS INJURIES: 
 Minor.
MEDICATIONS: 
 None.


PREVIOUS OPERATIONS: 
 None.
ALLERGIES:  
NONE KNOWN.
FAMILY HISTORY: 
 Negative for heart disease, hypertension, obesity, diabetes, cancer or stroke.
SOCIAL HISTORY: 
 The patient is single.  He is a student.  He does not smoke, drink alcohol or consume drugs.
REVIEW OF SYSTEMS
CONSTITUTIONAL:  The patient denies weight loss/gain, fever, chills.


ENMT:  The patient denies headaches, nosebleeds, voice changes, blurry vision, changes in/loss of vision.
CV:  The patient denies chest pain, SOB supine, palpitations, edema, varicose veins, leg pains.
RESPIRATORY: 